# 00 · Setup and data

Run this notebook **once per Colab session**. It:

1. checks that you have a GPU,
2. mounts your Google Drive (everything important is saved there),
3. clones this repository and installs the requirements,
4. downloads LOL-v1, LOL-v2 (Real + Synthetic) and the unpaired test sets (DICM, LIME, MEF, NPE, VV),
   **or** restores them from the zip cache on your Drive if they were downloaded before,
5. verifies the image pairs, then shows a summary table and preview images.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
When Colab asks for permission to access Google Drive, allow it.

The first run downloads about 1 GB and takes a few minutes. Later runs only unzip the cache from Drive and take under a minute.

## 1 · Settings

In [ ]:
REPO_URL = "https://github.com/Asad3404/LUMA.git"
BRANCH = "claude/attention-unet-low-light-ay0gpv"   # change to "main" once this branch is merged

DRIVE_ROOT = "/content/drive/MyDrive/lowlight-attn-unet"   # checkpoints, logs, results, dataset cache
DATA_ROOT = "/content/data"                                # fast local disk used for training
REPO_DIR = "/content/LUMA"

## 2 · Check the GPU
You should see a **Tesla T4**. If you get an error instead, set the runtime type to GPU (see the top of this notebook).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 3 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
for sub in ["datasets/manual", "runs", "results"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
print("Drive folder ready:", DRIVE_ROOT)

## 4 · Get the code and install requirements
Running this again pulls the latest version of the code.

In [ ]:
import os
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="code version: %h  %s"
# torch/torchvision are already on Colab; pip keeps those versions.
!pip install -q -r requirements.txt

## 5 · Download (or restore) the datasets

Here is what the script does:
* If `MyDrive/lowlight-attn-unet/datasets/<name>.zip` already exists, it unzips that copy. Nothing is downloaded again.
* Otherwise it tries Hugging Face first and Google Drive second. It then reorganises the files into
  `/content/data/<name>/{train,test}/{low,high}`, checks them, and saves a zip to your Drive for next time.

**If you see `GOOGLE DRIVE REFUSED THE DOWNLOAD`:** Google limits how often popular public files can be downloaded each day.
Follow the printed steps (download the file in your browser, upload it to `MyDrive/lowlight-attn-unet/datasets/manual/`) and run this cell again.
To download a single dataset, use for example `--datasets lol_v1`.

In [ ]:
!python data/download.py --datasets all \
    --data_root {DATA_ROOT} \
    --cache_dir {DRIVE_ROOT}/datasets \
    --results_dir {DRIVE_ROOT}/results/data

## 6 · Look at the results
The summary JSON and preview grids are saved on your Drive in `results/data/`. Each grid shows low-light test images on the top row and ground truth on the bottom row.

In [ ]:
import json, glob
from IPython.display import Image, display

summary = json.load(open(f"{DRIVE_ROOT}/results/data/data_summary.json"))
for rep in summary["paired"]:
    for split, s in rep.get("splits", {}).items():
        print(f"{rep['name']:<18} {split:<5} pairs={s['pairs']:>4} (expected {s['expected']})  sizes={s['sizes']}")
    print(f"{'':<18} source: {rep.get('source', 'FAILED: ' + str(rep.get('problems')))}")
if summary["unpaired"]:
    print("unpaired:", summary["unpaired"]["counts"], "missing:", summary["unpaired"]["missing"])
print("LOL-v1 / LOL-v2-Real overlap:", summary.get("lol_v1_v2_overlap"))

for png in sorted(glob.glob(f"{DRIVE_ROOT}/results/data/preview_*.png")):
    print(os.path.basename(png))
    display(Image(png))

## 7 · Sanity check: one training batch
This loads a batch of random 256×256 crops (with flips and rotations) and one full-resolution test image.

In [ ]:
import sys, yaml, torch
import matplotlib.pyplot as plt
sys.path.insert(0, REPO_DIR)
from src.datasets import build_loaders

cfg = yaml.safe_load(open("configs/base.yaml"))
cfg["data"]["root"] = DATA_ROOT
train_loader, val_loader = build_loaders(cfg, seed=cfg["seed"])
batch = next(iter(train_loader))
val = next(iter(val_loader))
print("train batch:", tuple(batch["low"].shape), "| test image:", tuple(val["low"].shape),
      "| train iters/epoch:", len(train_loader))

fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    ax[0, i].imshow(batch["low"][i].permute(1, 2, 0)); ax[0, i].set_title("low (crop)")
    ax[1, i].imshow(batch["high"][i].permute(1, 2, 0)); ax[1, i].set_title("high (crop)")
for a in ax.flat: a.axis("off")
plt.tight_layout(); plt.show()

## 8 · Unit tests
These should all pass. They run offline on tiny fake datasets.

In [ ]:
!python -m pytest tests -q

### Done ✅
If every dataset in step 5 shows `OK`, Phase 1 is complete. In a new Colab session, re-running this notebook restores the data from Drive in under a minute.